# Error Analysis for CCMT Model

This notebook analyzes model predictions and errors:
- Load trained model and predictions
- Visualize confusion matrix
- Analyze per-class performance
- Identify common error patterns

In [ ]:
import sys
sys.path.append('..')

import os
import yaml
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader

from data.dataset import MSP_Podcast_Dataset, collate_fn
from models.full_model import CCMTModel
from utils.metrics import MetricsTracker, plot_confusion_matrix

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Configuration and Model

In [ ]:
# Load config
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load model
model = CCMTModel(
    audio_backbone=config['model']['audio_backbone'],
    text_en_backbone=config['model']['text_en_backbone'],
    text_fr_backbone=config['model']['text_fr_backbone'],
    num_classes=config['model']['num_classes'],
    hidden_dim=config['model']['hidden_dim'],
    num_attention_heads=config['model']['num_attention_heads'],
    dropout=config['model']['dropout'],
    token_sample_k=config['model']['token_sample_k']
).to(device)

# Load checkpoint
checkpoint_path = '../checkpoints/best_model.pth'
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded checkpoint from epoch {checkpoint['epoch']}")
else:
    print(f"Checkpoint not found: {checkpoint_path}")

## 2. Load Test Dataset and Generate Predictions

In [ ]:
# Create test dataset
test_dataset = MSP_Podcast_Dataset(
    audio_root=os.path.join(config['data']['dataset_root'], 'Audios', 'Audios'),
    labels_csv=os.path.join(config['data']['dataset_root'], 'Labels', 'labels_consensus.csv'),
    transcripts_en_file='../data/transcripts/transcripts_en.json',
    transcripts_fr_file='../data/cache/translations_en_ro.json',
    partition='Test1',
    target_sample_rate=config['data']['sample_rate']
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)

print(f"Test dataset size: {len(test_dataset)}")

In [ ]:
# Generate predictions
model.eval()
all_predictions = []
all_targets = []
all_file_ids = []

with torch.no_grad():
    for batch in test_loader:
        audio = batch['audio'].to(device)
        text_en = batch['text_en']
        text_fr = batch['text_fr']
        labels = batch['labels'].to(device)
        file_ids = batch['file_ids']
        
        logits = model(audio, text_en, text_fr)
        predictions = logits.argmax(dim=-1)
        
        all_predictions.extend(predictions.cpu().numpy())
        all_targets.extend(labels.cpu().numpy())
        all_file_ids.extend(file_ids)

all_predictions = np.array(all_predictions)
all_targets = np.array(all_targets)

print(f"Generated predictions for {len(all_predictions)} samples")

## 3. Visualize Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix

class_names = ['Angry', 'Happy', 'Sad', 'Neutral']
cm = confusion_matrix(all_targets, all_predictions)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

# Normalize confusion matrix
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
plt.figure(figsize=(10, 8))
sns.heatmap(cm_normalized, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Normalized Confusion Matrix - Test Set')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

## 4. Per-Class Performance

In [ ]:
from sklearn.metrics import classification_report

# Generate classification report
report = classification_report(all_targets, all_predictions,
                              target_names=class_names, digits=4)
print("\nClassification Report:")
print(report)

# Create DataFrame for visualization
from sklearn.metrics import precision_score, recall_score, f1_score

metrics_per_class = []
for i, class_name in enumerate(class_names):
    precision = precision_score(all_targets == i, all_predictions == i)
    recall = recall_score(all_targets == i, all_predictions == i)
    f1 = f1_score(all_targets == i, all_predictions == i)
    
    metrics_per_class.append({
        'Class': class_name,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })

df_metrics = pd.DataFrame(metrics_per_class)
print("\nPer-Class Metrics:")
print(df_metrics)

In [ ]:
# Plot per-class metrics
df_metrics.set_index('Class')[['Precision', 'Recall', 'F1-Score']].plot(kind='bar', figsize=(12, 6))
plt.title('Per-Class Performance Metrics')
plt.ylabel('Score')
plt.xlabel('Emotion Class')
plt.legend(loc='lower right')
plt.xticks(rotation=45)
plt.ylim([0, 1])
plt.tight_layout()
plt.show()

## 5. Identify Common Errors

In [ ]:
# Find misclassified samples
errors = all_predictions != all_targets
error_indices = np.where(errors)[0]

print(f"\nTotal errors: {len(error_indices)} / {len(all_targets)} ({len(error_indices)/len(all_targets)*100:.2f}%)")

# Analyze error patterns
error_patterns = {}
for idx in error_indices:
    true_label = class_names[all_targets[idx]]
    pred_label = class_names[all_predictions[idx]]
    pattern = f"{true_label} → {pred_label}"
    
    if pattern not in error_patterns:
        error_patterns[pattern] = []
    error_patterns[pattern].append(all_file_ids[idx])

# Sort by frequency
error_patterns_sorted = sorted(error_patterns.items(), key=lambda x: len(x[1]), reverse=True)

print("\nMost common error patterns:")
for pattern, files in error_patterns_sorted[:10]:
    print(f"  {pattern}: {len(files)} errors")

## 6. Error Analysis by Pattern

In [ ]:
# Visualize error patterns
error_counts = {pattern: len(files) for pattern, files in error_patterns.items()}
patterns = list(error_counts.keys())
counts = list(error_counts.values())

plt.figure(figsize=(14, 6))
plt.bar(range(len(patterns)), counts, color='coral')
plt.xticks(range(len(patterns)), patterns, rotation=90)
plt.xlabel('Error Pattern (True → Predicted)')
plt.ylabel('Number of Errors')
plt.title('Error Pattern Distribution')
plt.tight_layout()
plt.show()

## 7. Examine Specific Error Cases

In [ ]:
# Show details of some error cases
print("\nExample error cases:")
print("="*80)

for i, idx in enumerate(error_indices[:5]):
    file_id = all_file_ids[idx]
    true_label = class_names[all_targets[idx]]
    pred_label = class_names[all_predictions[idx]]
    
    print(f"\nError {i+1}:")
    print(f"  File: {file_id}")
    print(f"  True label: {true_label}")
    print(f"  Predicted: {pred_label}")
    print("-" * 80)

## 8. Summary

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

accuracy = accuracy_score(all_targets, all_predictions)
macro_f1 = f1_score(all_targets, all_predictions, average='macro')
weighted_f1 = f1_score(all_targets, all_predictions, average='weighted')

print("\n" + "="*80)
print("ERROR ANALYSIS SUMMARY")
print("="*80)
print(f"Test set size: {len(all_targets)}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Macro-F1: {macro_f1:.4f}")
print(f"Weighted-F1: {weighted_f1:.4f}")
print(f"\nTotal errors: {len(error_indices)} ({len(error_indices)/len(all_targets)*100:.2f}%)")
print(f"Most common error: {error_patterns_sorted[0][0]} ({len(error_patterns_sorted[0][1])} cases)")
print("="*80)